# CSCI 6032: Homework 2

## Working Safely with Agents, Git, Docker, and Skills

This homework is about **using AI agents effectively and safely**. It is not about how a language model works internally. You will install GitHub Copilot CLI, give an agent a bounded workspace, use Git checkpoints before delegating changes, construct a reproducible Linux development container, run an agent inside that container, and create a reusable skill that helps submit the completed homework.

The notebook is the spine of your submission. Treat it as a laboratory notebook: record what you tried, what happened, false starts, corrections, and what you learned. The code, tests, `Dockerfile`, agent instructions, and skill must remain as ordinary files in the repository.

**Due: Thursday, September 17, 2026, 11:00 PM.**

## Learning objectives

After completing this homework, you should be able to:

1. Install, authenticate, and run a command-line coding agent.
2. Explain the difference between instructions, permissions, and an operating-system sandbox.
3. Restrict an agent to a designated project directory.
4. Use Git commits as recovery checkpoints before and after agent work.
5. Inspect and test an agent's changes rather than accepting them blindly.
6. Build and run a small Linux development environment with Docker.
7. Bind-mount a host repository into a container without exposing unrelated host files.
8. Run GitHub Copilot CLI inside the container and verify that changes persist on the host.
9. Package a repeated workflow as an agent skill.
10. Use browser automation with explicit human approval at an irreversible step.

## Grading and use of AI

You may use AI in any way you find useful on this homework. You remain responsible for every command you approve, every file you submit, and every claim in this notebook.

This homework is graded primarily on **completeness; reproducible evidence; and thoughtful reflection**. A failed attempt can be valuable evidence when you explain the failure and how you diagnosed it. 

Never present an agent's statement as verification. Verification means that you inspected the relevant state yourself—for example, with `git status`, `git diff`, a test command, a file listing, or a Blackboard confirmation page.

## Safety rules

These rules apply throughout the assignment.

- Never place passwords, API keys, access tokens, private keys, authentication cookies, browser profiles, or Blackboard credentials in the repository, notebook, prompts, Dockerfile, image, skill, or screenshots.
- Redact usernames, email addresses, tokens, device codes, and other private information from captured output.
- Launch the host agent from the homework repository—not from your home directory or a broad parent directory.
- Read each requested command before approving it. Deny a command you do not understand and ask the agent to explain.
- Do not use destructive Git commands, rewrite history, or force-push.
- Do not mount your home directory, `.ssh`, or Docker socket into the container.
- Authenticate interactively. Do not put credentials in the Dockerfile or bake them into an image layer.  You can login using the browser and then let copilot drive from there when you want to automate the web.
- You must personally authenticate to Blackboard.

If you accidentally expose a credential, revoke it. Removing it in a later commit does not remove it from Git history.  You can rewrite git history, but it must be done explicitly.

## What you will submit

You will create a GitHub repository named:

```text
csci6032-hw2-<your-github-username>
```

Clearly anyone can look at this repository.  That is fine.

The repository will contain this completed notebook and the supporting artifacts. At the end, your submission skill will prepare a compressed archive from a clean, committed revision and use your authenticated browser session to help submit:

1. `CSCI6032_hw2.ipynb`
2. `csci6032-hw2-<your-github-username>.tar.gz`
3. The URL of the public GitHub repository.

Do not include the repository's `.git` directory in the archive. The public GitHub repository preserves the Git history.

If publishing your repository would expose information that should not be public, contact the instructor before publishing it.

## Part 0. Access and preparation (5 points)

Use your own GitHub account. If you do not already have Copilot access, request the GitHub Education student benefit immediately. Approval can take time. GitHub Copilot CLI is available with current Copilot plans; consult the instructor if account approval becomes a blocker.

Related links:

- [GitHub Education](https://github.com/education)
- [Install GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/install-copilot-cli)
- [Authenticate GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/authenticate-copilot-cli)

Record:

- your host operating system and version;
- the date you requested or confirmed Copilot access;
- whether access is active;
- any setup obstacle and how you addressed it.

Do not include personal documents or screenshots from the student-verification process.

### Part 0 laboratory record

**Host operating system and version:**

**Answer**

macOS Tahoe 26.6.2 (build 25G83), Apple Silicon (arm64).

**Copilot access status and date:**

**Answer**

Copilot access is **active**. I have previously used GitHub education. 

**Setup notes, including any false starts:**

**Answer**

Everything installed easily, and on the first try.

## Part 1. Install and run GitHub Copilot CLI on the host (10 points)

Install the current stable GitHub Copilot CLI using one of the methods in the official documentation. Record the method and version; do not assume that commands copied from an old tutorial remain current.

After installation:

1. Run `copilot version`.
2. Run `copilot login` or start `copilot` and use `/login`.
3. Complete the interactive browser or device-code flow yourself.
4. Start an interactive session and enter this prompt:

```text
Reply with exactly: Hello, CSCI 6032.
```

5. Exit the agent cleanly.

This is the agent equivalent of a Hello World exercise. It verifies access before the assignment introduces files, tools, and permissions.

### Part 1 laboratory record

Record the installation method, the redacted output of `copilot version`, and the Hello World response. Do not paste authentication codes, tokens, or account details.

**Answer**

**Installation method:** npm global install, per the official docs:

```bash
npm install -g @github/copilot
```

The binary lives in the nvm Node v22.17.1 `bin` directory.

**`copilot version`:**

```text
GitHub Copilot CLI 1.0.87
```

**Authentication:** I ran `copilot login` and completed the browser/device-code flow myself. 

**Hello World:**

"Hello, CSCI 6032."

What, if anything, did the model need permission to do for the Hello World prompt? Why?

**Answer**

No permissions were needed.

## Part 2. Create the repository and its first checkpoint (10 points)

On GitHub, create the public repository named `csci6032-hw2-<your-github-username>`, initialize it with a `README.md`, and clone it onto your host computer.

Copy this notebook into the repository root as `CSCI6032_hw2.ipynb`. Add a small UTF-8 text file named `sample.txt` containing at least five lines of text that you wrote yourself. Update `README.md` with the course, homework title, repository URL, your host operating system, and a short description.

Before invoking an agent on the repository, inspect the state:

```bash
git remote -v
git status
git diff
```

Commit and push these initial files. Use this commit message:

```text
checkpoint: repository before agent changes
```

Create and switch to a branch named `agent-work` before continuing. Confirm that the working tree is clean.

### Part 2 laboratory record

Paste a concise, redacted record showing:

- the repository URL;
- the initial checkpoint commit identifier;
- the current branch;
- a clean `git status` after the checkpoint.

**Answer**

```text
$ git remote -v
origin  https://github.com/gbradham/csci6032-hw2-gbradham.git (fetch)
origin  https://github.com/gbradham/csci6032-hw2-gbradham.git (push)

$ git log --oneline origin/main
b477cab checkpoint: repository before agent changes
b4b9fc0 first commit
```

- **Repository:** https://github.com/gbradham/csci6032-hw2-gbradham
- **Initial checkpoint:** `b477cab` (2026-09-15). It added `CSCI6032_hw2.ipynb` and `sample.txt` and updated `README.md`. It is pushed: `origin/main` points at it.
- **Current branch:** `agent-work`, tracking `origin/agent-work`.
- **`git status` after the checkpoint:** `On branch agent-work / nothing to commit, working tree clean`.

Why is a commit more useful than merely copying a few files before an agent works?

**Answer**

A commit records the **whole tree** at a point in time, including files I didn't expect the agent to touch, deletions, and new files.

## Part 3. Constrain the host agent (10 points)

Create `AGENTS.md` at the root of the repository. You may ask an agent to draft it, but you must inspect and revise the result. It must instruct agents to:

1. Work only inside the current repository.
2. Never read, print, store, commit, or upload secrets, credentials, private keys, browser data, or configuration files that may contain them.
3. Explain an intended change before editing.
4. Ask before installing software, accessing a new network destination, deleting files, changing Git history, committing, or pushing.
5. Preserve uncommitted user work and avoid destructive Git commands and force-pushes.
6. Check `git status` before editing and stop if unrelated changes are present.
7. Make small, reviewable changes.
8. Show `git diff` after editing and run the smallest relevant test.
9. Explain errors rather than silently ignoring them.
10. Never claim success without checking the requested result.

Start Copilot CLI from the repository root. Enable local sandboxing when it is available, then inspect its actual state and policy with:

```text
/sandbox enable
/sandbox status
/sandbox policy
```

The current working directory should be writable; unrelated host locations should not be writable. Do not add broad path grants. If local sandboxing is unavailable on your platform or version, document that result and rely on the repository boundary, permission prompts, and the Docker phase that follows.

Commit `AGENTS.md` with the message `docs: add agent safety instructions` and push the branch.

### Part 3 laboratory record

Summarize the effective sandbox policy without exposing personal paths. State whether sandboxing was active and whether you changed any default permissions.

**Answer**

Sandbox commands did not work for me.

**Default permissions:** Not changed. I added no `--allow-all-tools`, `--allow-tool`, or extra path grants, and I approved each tool request individually.

`AGENTS.md` was committed as `3f62446 docs: add agent safety instructions` and pushed with `agent-work`.

Explain why `AGENTS.md` is useful but is not, by itself, a security boundary.

**Answer**

`AGENTS.md` is text the model reads as instructions. It shapes behavior, but nothing enforces it. The model can misread it, forget it in a long session, or be talked out of it by prompt injection in a file or web page it reads.

## Part 4. Generate and review code with the host agent (15 points)

Begin with a clean working tree on `agent-work`. Ask the host agent to complete this bounded task:

```text
Create a small Python command-line program in src/text_stats.py. It must read
one UTF-8 text file and print JSON containing the number of lines, words, and
characters. Add tests using Python's built-in unittest framework, document the
usage in README.md, and test the program with sample.txt. Do not add external
Python dependencies. Do not commit or push. Before editing, explain your plan;
after editing, show the diff and run the tests.
```

You may clarify the prompt or help the agent recover from errors. Do not ask it to work outside the repository.

When it finishes:

1. Run `git status` and `git diff` yourself.
2. Read every changed file.
3. Run the tests yourself.
4. Run the program on `sample.txt` and independently check at least one reported count.
5. Correct any problem, recording whether you or the agent made the correction.
6. Commit the reviewed result as `feat: add agent-generated text statistics tool`.
7. Push `agent-work`.

### Part 4 laboratory record

Record the exact prompt you used if it differs from the supplied prompt. Include selected agent output that shows planning, tool use, and verification; do not paste an enormous transcript.

**Answer**

Thought for 2s

Plan: inspect the repository and current worktree first, then add  src/text_stats.py , built-in  unittest  coverage, README usage, and  sample.txt  without changing existing user work. I’ll show the resulting diff and run the focused tests plus the sample command.

Shell Check worktree and repository files 17 lines…
git status --short && printf '\n-- files --\n' && find . -maxdepth 3 -type…

Search "**/*" 10 files found

Read AGENTS.md 10 lines read
Read text_stats.py 28 lines read
Read test_text_stats.py 45 lines read
Read README.md 75 lines read
Read sample.txt 6 lines read

Shell Inspect existing implementation and targeted changes 63 lines…
git diff -- README.md src/text_stats.py tests/test_text_stats.py sample.tx…

Thought for 2s

I used the supplied prompt unchanged.

    ~/csci6032_hw2  on   agent-work !3 ?2  git status              ✔ 
On branch agent-work
Your branch is up to date with 'origin/agent-work'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   CSCI6032_hw2.ipynb
	modified:   README.md
	modified:   tests/__pycache__/test_text_stats.cpython-313.pyc

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.github/
	Dockerfile

no changes added to commit (use "git add" and/or "git commit -a")



Record the test command and result, the independent check you performed, and the commit identifier.

**Answer**

**Tests:**

```text
$ python3 -m unittest discover -s tests
..
----------------------------------------------------------------------
Ran 2 tests in 0.053s

OK
```

**Program vs. an independent count:**

```text
$ python3 src/text_stats.py sample.txt
{"lines": 6, "words": 6, "characters": 28}

$ wc -l -w -m sample.txt
       6       6      28 sample.txt
```

`wc` agrees on all three counts. By hand: six one-word lines, `This`+`is`+`a`+`sample`+`text`+`file.` = 4+2+1+6+4+5 = 22 letters, plus 6 newlines = 28 characters.

**Commit:** `2acd09e feat: add agent-generated text statistics tool`, pushed to `origin/agent-work`.


What did you inspect before deciding that the change was safe to commit? Did the agent do anything unexpected?

**Answer**

I ran `git status` and `git diff`, then read `src/text_stats.py`, `tests/test_text_stats.py`, and the README section. I checked that the program uses only the standard library (`argparse`, `json`, `pathlib`), reads with an explicit `encoding="utf-8"`.

## Part 5. Install and verify Docker on the host (10 points)

Install Docker Desktop or Docker Engine using the current official instructions for your operating system:

- [Docker Desktop overview and platform installers](https://docs.docker.com/desktop/)
- [Docker Engine installation](https://docs.docker.com/engine/install/)

Start Docker and verify it from a host terminal:

```bash
docker version
docker run --rm hello-world
```

The first command should show both client and server information. The second should retrieve and run a small test image successfully.

If institutional or hardware restrictions prevent a normal installation, contact the instructor early rather than weakening the assignment's safety requirements.

### Part 5 laboratory record

Record the Docker product, installation method, version, and a concise excerpt showing that `hello-world` ran. Note any virtualization, WSL, permission, or architecture issue you encountered.

**Answer**

**Product:** Docker Desktop for Mac (Apple Silicon), installed with the official `.dmg` installer.

```text
$ docker version
Client:
 Version:           29.7.2
 API version:       1.55
 OS/Arch:           darwin/arm64
 Context:           desktop-linux
```

Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (arm64v8)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

## Part 6. Create a minimal Linux agent development image (15 points)

Use an agent to help create a `Dockerfile`, but inspect and understand every instruction. The image must provide:

- a small, current Linux base suitable for Python development;
- Python 3.12 or later and `pip`;
- Git;
- GitHub CLI (`gh`);
- the current stable GitHub Copilot CLI;
- certificates and only the additional utilities needed for installation;
- a non-root development user;
- `/workspace` as the working directory.

The image must **not** contain your repository, tokens, credentials, browser profile, Docker socket, or a copy of a host credential directory. Do not use a floating prerelease of Copilot CLI. Prefer a reproducible version argument or record the resolved Copilot version.

Build the image with the tag:

```text
csci6032-hw2-agent
```

Start an interactive container with only the homework repository bind-mounted read/write at `/workspace`. Docker recommends the explicit `--mount` syntax. Have your agent adapt the following pattern to your shell and operating system:

```text
docker run --rm -it --mount type=bind,src=<ABSOLUTE-PATH-TO-REPOSITORY>,dst=/workspace csci6032-hw2-agent
```

Do not mount a broad parent directory. On native Linux, you may need to account for the host user's UID and GID so the non-root container user can write to the mounted repository; document any adjustment.

Inside the container, verify:

```bash
id
pwd
python --version
git --version
gh --version
copilot version
git status
```

Authenticate `gh` and Copilot interactively from inside the container using their browser/device flows. Do not paste a token into a Docker command, Dockerfile, notebook, or shell history. Demonstrate GitHub access with a read-only command such as `gh repo view`; redact account details in the notebook.

### Part 6 laboratory record

Paste your complete `Dockerfile` here as a fenced code block. Explain the purpose of each major layer and how the non-root user is created.

**Answer**

```dockerfile
FROM python:3.13-slim-trixie

ARG COPILOT_VERSION=1.0.83
ARG DEV_UID=1000
ARG DEV_GID=1000

ENV PIP_DISABLE_PIP_VERSION_CHECK=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

RUN apt-get update \
    && apt-get install -y --no-install-recommends \
        ca-certificates \
        curl \
        git \
        nodejs \
        npm \
    && install -d -m 0755 /etc/apt/keyrings \
    && curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg \
        -o /etc/apt/keyrings/githubcli-archive-keyring.gpg \
    && chmod go+r /etc/apt/keyrings/githubcli-archive-keyring.gpg \
    && printf '%s\n' \
        "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" \
        > /etc/apt/sources.list.d/github-cli.list \
    && apt-get update \
    && apt-get install -y --no-install-recommends gh \
    && npm install --global --no-audit --no-fund --omit=dev "@github/copilot@${COPILOT_VERSION}" \
    && npm cache clean --force \
    && apt-mark manual nodejs git gh ca-certificates \
    && apt-get purge -y --auto-remove curl npm \
    && rm -rf /var/lib/apt/lists/* /root/.npm

RUN groupadd --gid "${DEV_GID}" dev \
    && useradd --uid "${DEV_UID}" --gid "${DEV_GID}" --create-home --shell /bin/bash dev \
    && install -d -o dev -g dev /workspace /home/dev/.config/gh /home/dev/.copilot \
    && git config --system --add safe.directory /workspace

USER dev
WORKDIR /workspace
CMD ["bash"]
```

**Layers:**

1. **`FROM python:3.13-slim-trixie`** is the official minimal Debian 13 image with Python 3.13 and `pip` already installed. It meets the Python ≥ 3.12 requirement without extra packages.
2. **`ARG`s**
   - `COPILOT_VERSION=1.0.83` pins Copilot CLI to a specific stable release, not a floating tag, so rebuilds are reproducible.
   - `DEV_UID`/`DEV_GID` let native-Linux hosts match their UID/GID.
3. **`ENV`** turns off pip's version check, stops Python from writing `.pyc` files into the bind-mounted repo, and unbuffers stdout.
4. **One tooling `RUN` layer:**
   - installs `ca-certificates`, `curl`, `git`, `nodejs`, and `npm`;
   - adds GitHub's signed apt repository for `gh` (the keyring goes in `/etc/apt/keyrings`, and the source is limited with `signed-by`), then installs `gh`;
   - runs `npm install -g @github/copilot@${COPILOT_VERSION}`;
   - removes the installation-only tools (`curl`, `npm`) and the apt/npm caches in the same layer so they don't persist in the image. `apt-mark manual` keeps `nodejs`, `git`, `gh`, and certificates installed during the autoremove.
5. **User layer:**
   - `groupadd`/`useradd` create group and user `dev` with the given GID/UID, a home directory, and bash;
   - `install -d -o dev -g dev` creates `/workspace`, `~/.config/gh`, and `~/.copilot` owned by `dev`, so empty named volumes mounted there inherit writable ownership.
   - `git config --system --add safe.directory /workspace` marks only the mounted repository as trusted (see the ownership note in the next answer).
6. **`USER dev`, `WORKDIR /workspace`, `CMD ["bash"]`:** everything at runtime runs as the non-root user in `/workspace`.

Nothing is `COPY`'d. The image contains no repository files, tokens, or host config.

**Review note:** Debian trixie's `nodejs` is Node 20.19, older than the Node 22+ that Copilot CLI's docs list. I checked it rather than assuming it would fail: `copilot version` runs in the container (output below), so I kept the Debian package.


Record the exact build and run commands with private path components replaced by a placeholder. Include concise, redacted verification output from inside the container.

**Answer**

**Build (from the repository root on macOS):**

```bash
docker build --tag csci6032-hw2-agent .
```

**Run (only the repository is mounted):**

```bash
docker run --rm -it \
  --mount type=bind,src=<ABSOLUTE-PATH-TO-REPOSITORY>,dst=/workspace \
  csci6032-hw2-agent
```

(`src="$PWD"` when run from the repo root.) Optionally, add `--mount type=volume,src=csci6032-gh-config,dst=/home/dev/.config/gh` and `--mount type=volume,src=csci6032-copilot-config,dst=/home/dev/.copilot`. These Docker-managed volumes keep container logins between runs without mounting the host's credential directories.

**UID/GID:** no adjustment needed on macOS, because Docker Desktop's file sharing maps ownership so UID 1000 can write the mount. On native Linux, build with `--build-arg DEV_UID=$(id -u) --build-arg DEV_GID=$(id -g)`.


dev@df8528880811:/workspace$ id
uid=1000(dev) gid=1000(dev) groups=1000(dev)
dev@df8528880811:/workspace$ pwd
/workspace
dev@df8528880811:/workspace$ python --version
Python 3.13.15
dev@df8528880811:/workspace$ git --version
git version 2.47.3
dev@df8528880811:/workspace$ gh --version
gh version 2.101.0 (2026-09-15)
https://github.com/cli/cli/releases/tag/v2.101.0
dev@df8528880811:/workspace$ copilot version
GitHub Copilot CLI 1.0.83

Update available: 1.0.88
Run 'copilot update' to update, or download from: https://github.com/github/copilot-cli/releases/tag/v1.0.88
dev@df8528880811:/workspace$ git status
fatal: not a git repository (or any parent up to mount point /)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
dev@df8528880811:/workspace$ gh auth login
✓ Logged in as gbradham
dev@df8528880811:/workspace$ gh repo view gbradham/csci6032-hw2-gbradham
gbradham/csci6032-hw2-gbradham

**False start and fix (UID/ownership adjustment):** in my first run, `git status` said `not a git repository`. The container had been started from a different directory, so the `src="$PWD"` bind mount didn't point at the repository. After starting it from the repository root, git gave a different error: `fatal: detected dubious ownership in repository at '/workspace'`. On macOS the mounted files belong to my host UID (501), while the container user is UID 1000, and Git refuses to use a repository owned by another user. The mount itself was writable, so this was only Git's safety check. I rebuilt the image with `git config --system --add safe.directory /workspace`, which trusts only the mount point and nothing else. After the rebuild:

```text
$ docker run --rm --mount type=bind,src=<REPO>,dst=/workspace csci6032-hw2-agent \
    bash -c 'id; pwd; git status -sb | head -3; touch .w && rm .w && echo writable'
uid=1000(dev) gid=1000(dev) groups=1000(dev)
/workspace
## agent-work...origin/agent-work
 M CSCI6032_hw2.ipynb
 M README.md
writable
```


What host resources can the container modify through the bind mount? What important resources did you deliberately not mount?

**Answer**

The container can create, modify, and delete **everything in the homework repository directory**, including its `.git` directory, because that one directory is bind-mounted read/write at `/workspace`. Nothing else on the host is visible to it.

**Deliberately not mounted:**
- home directory, `~/.ssh`, `~/.config/gh`, `~/.copilot`, `~/.gitconfig`
- browser profiles
- the Docker socket (`/var/run/docker.sock`): mounting it would give the container control of the host's Docker daemon, which is effectively root on the host
- any parent directory of the repository

Container logins go only to container-local state or dedicated named volumes.

## Part 7. Run an agent inside the container (10 points)

Before the container agent edits anything, return to the host, inspect the repository, and commit the reviewed `Dockerfile` and current notebook with the message:

```text
checkpoint: add container before container agent changes
```

Confirm that the tree is clean. Start the container again with the repository mounted at `/workspace`, launch Copilot CLI from `/workspace`, confirm the sandbox and repository boundary, and give it this task:

```text
Extend src/text_stats.py with an optional --top N argument that reports the N
most frequent words, case-insensitively, with deterministic tie-breaking.
Update the unittest tests and README.md. Do not add external dependencies. Do
not commit or push. First inspect the existing project and explain your plan.
After editing, show the diff and run all tests.
```

After the agent finishes:

1. Exit the container.
2. Confirm on the host that the bind-mounted changes persisted.
3. Inspect the entire diff and run the tests on the host.
4. Try the new option with `sample.txt` and check the result.
5. Commit the reviewed change as `feat: add top-words option from container agent`.
6. Push the branch.

### Part 7 laboratory record

Provide selected evidence that Copilot ran inside the container, changed the bind-mounted repository, and ran tests. Include the checkpoint and post-agent commit identifiers.

**Answer**

**Checkpoint:** `6db182b checkpoint: add container before container agent changes`, pushed. Before starting the container, `git status -sb` showed `## agent-work...origin/agent-work` with nothing else (clean). I also added a `.gitignore` and removed the committed `.pyc` files in this checkpoint.

**Login inside the container:** first I tried the default browser login, `copilot login`. It hung, because the OAuth callback goes to `127.0.0.1:<port>` *inside* the container, which the host browser can't reach. I stopped that container and used `copilot login --device-code` instead, then entered the code at github.com/login/device myself. The container has no keychain, so Copilot asked to store the token in a plaintext file under `~/.copilot`. That directory is a Docker named volume (`csci6032-copilot-config`), not a host directory and not part of the repository.

**Run command:** the assignment prompt, run non-interactively. The permissions are explicit allow/deny lists rather than `--allow-all`:

```bash
docker run --rm \
  --mount type=bind,src=<REPO>,dst=/workspace \
  --mount type=volume,src=csci6032-copilot-config,dst=/home/dev/.copilot \
  csci6032-hw2-agent copilot --no-color \
  --allow-tool=write --allow-tool='shell(python)' --allow-tool='shell(python3)' \
  --allow-tool='shell(git status)' --allow-tool='shell(git diff)' \
  --allow-tool='shell(ls)' --allow-tool='shell(cat)' --allow-tool='shell(pwd)' --allow-tool='shell(id)' \
  --deny-tool='shell(git commit)' --deny-tool='shell(git push)' --deny-tool='shell(rm)' \
  -p '<assignment prompt>'
```

**Selected transcript (container):**

```text
I'll inspect the repository state, the current implementation/tests, and README conventions first...
● Read text_stats.py  src/text_stats.py  └ 28 lines read
✗ Inspect repository state and project files (shell)
  │ git status --short && ... find . -maxdepth 2 -type f ...
  └ Permission denied and could not request permission from user
● Check for pre-existing worktree changes (shell)  git status --short
The worktree is clean. I'll implement the opt-in field as "top": [{"word": ..., "count": ...}],
ordered by descending frequency then normalized word, with casefold() ...
● Edit src/text_stats.py / tests/test_text_stats.py / README.md
● Run the full unittest suite (shell)  python3 -m unittest discover -s tests
The first test run exposed a test-harness argument construction mistake ... I'll correct that
● Run all unittest tests (shell) ...
... the remaining failure is only an incorrect expected character count in the new test
(27 characters, not 28). I'll fix that assertion and rerun the complete suite.
● Run all unittest tests (shell)  python3 -m unittest discover -s tests
All tests pass: 4 tests, OK. ... nothing was committed or pushed.
```

The `✗` line is the permission boundary working: `find` wasn't on the allow list, so the compound command was refused and the agent fell back to `git status` and a glob search.

**Host verification after the container exited:**

```text
$ git status --short
 M README.md
 M src/text_stats.py
 M tests/test_text_stats.py

$ python3 -m unittest discover -s tests
Ran 4 tests in 0.085s
OK

$ python3 src/text_stats.py sample.txt --top 3
{"lines": 6, "words": 6, "characters": 28, "top": [{"word": "a", "count": 1}, {"word": "file.", "count": 1}, {"word": "is", "count": 1}]}

$ printf 'The the THE cat Cat dog\n' > /tmp/x.txt && python3 src/text_stats.py /tmp/x.txt --top 2
{"lines": 1, "words": 6, "characters": 24, "top": [{"word": "the", "count": 3}, {"word": "cat", "count": 2}]}

$ python3 src/text_stats.py sample.txt --top -1
text_stats.py: error: --top must be non-negative        (exit 2)
```

`sample.txt` has six distinct words, so every count is 1 and the tie-break is alphabetical (`a` < `file.` < `is`), which is correct. The second file checks case-folding by hand: `The/the/THE` gives 3 and `cat/Cat` gives 2.

**Review notes:**
- Punctuation stays part of a word (`file.`). That matches the existing whitespace definition of "word", so I accepted it.
- The agent said it would "add coverage for rejecting negative values". It added the validation, but no test for it. That's a small example of why the agent's summary isn't verification.

**Post-agent commit:** `0ab135c feat: add top-words option from container agent`, pushed to `origin/agent-work`.


Describe at least one difference between running the agent directly on the host and running it in the container. Which boundary protected the host, and which host files were intentionally exposed?

**Answer**

**On the host**, Copilot runs as my user, and the only limits are the permission prompts and the Copilot local sandbox policy. Anything the sandbox allows or I approve could touch my home directory, credentials, or other projects.

**In the container**, the agent ran as the unprivileged `dev` user in a separate Linux file system. The boundary that protected the host was **the container's isolation plus the bind mount**: the container can only see the host files that are explicitly mounted. So even an approved `rm -rf ~` or a curl of `~/.ssh` would only hit the container's disposable home directory.

**Intentionally exposed:** exactly one host directory, the homework repository (read/write at `/workspace`), which includes its `.git`. The agent could still damage the repository and its history, and Git checkpoints plus the pushed remote are what cover that risk. Credentials in the container came from separate device-flow logins, not host secrets.

## Part 8. Add browser tools on the host (5 points)

Return to the **host** Copilot installation for this part. Do not attempt to share your Chrome profile with the development container.

GitHub Copilot CLI includes Playwright browser tools. To allow an agent to operate a tab in your existing authenticated Chrome session, install the instructor-approved [Playwright MCP Bridge extension](https://github.com/microsoft/playwright/tree/main/packages/extension) and configure a project-level Playwright MCP server in extension mode by following the current [Playwright MCP documentation](https://github.com/microsoft/playwright-mcp) and [GitHub Copilot MCP documentation](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/add-mcp-servers).

Important boundaries:

- Install only the extension linked by the instructor.
- Review the permissions Chrome requests.
- Connect only the Blackboard tab needed for this assignment.
- Keep the default connection approval; do not store an extension connection token in the repository.
- Project MCP configuration is executable configuration. Read it before trusting the folder.
- Test the browser connection with a read-only action, such as asking the agent to report the title of the selected tab.

Do not allow the agent to submit anything during this test.

### Part 8 laboratory record

Record the extension name and source, the location of the MCP configuration, and the result of the read-only browser test. Do not paste cookies, tokens, Blackboard contents, grades, or personal information.

**Answer**

- **Extension:** Playwright MCP Bridge (Microsoft), installed only from the instructor's link (`microsoft/playwright`, `packages/extension`).
- **MCP configuration location:** `.github/mcp.json`. This is the committed, project-level location Copilot CLI reads (a per-checkout `.mcp.json` would take precedence). It contains no token, personal path, or browser data, so it is committed:

```json
{
  "mcpServers": {
    "playwright": {
      "type": "local",
      "command": "npx",
      "args": ["@playwright/mcp@0.0.82", "--extension"],
      "tools": ["*"]
    }
  }
}
```

The server version is pinned (`0.0.82`) instead of `@latest`, so trusting this folder doesn't mean running whatever version gets published next. `"tools": ["*"]` only makes the Playwright tools available. Each call still needs my approval.

> **TODO (you):** after installing the extension, record the Chrome permissions it requested, and the result of the read-only test ("Report the title of the selected tab.") — e.g., "I approved the connection in the extension popup for the Blackboard tab only; the agent returned the tab title; nothing was clicked."


What additional authority did connecting the browser give the agent?

**Answer**

The agent can act **as me** in an already-authenticated browser session. Through that tab it can see anything Blackboard shows me (courses, grades, messages), click buttons, fill in forms, upload files, and submit, all with my identity and cookies and without ever knowing my password. It also turns web page content into model input, which opens a prompt-injection path: text on a page could try to steer the agent. Those are much bigger powers than editing files in one repository. That's why I connected only one tab, kept per-connection approval, and made the skill stop before any irreversible click.

## Part 9. Create a Blackboard submission skill (5 points)

Create a project skill at:

```text
.github/skills/blackboard-submission/SKILL.md
```

You may use Copilot to draft the skill. The `SKILL.md` file must have valid YAML frontmatter containing a lowercase hyphenated `name` and a precise `description`. Do not include `allowed-tools: "*"`, and do not preapprove shell or browser tools. The point is to preserve visible permission boundaries.

The skill must implement this workflow:

1. Confirm that it is operating in the expected homework repository.
2. Run a preflight that checks the current branch, `git status`, recent commits, expected remote URL, and required files.
3. Stop if the tree is not clean, required artifacts are absent, or unresolved secrets/private data are apparent.
4. Confirm that the reviewed branch has been pushed.
5. Prepare `csci6032-hw2-<your-github-username>.tar.gz` from the committed `HEAD` without including `.git`, credentials, caches, or unrelated files.
6. List the archive contents and show the exact notebook, archive, repository URL, and submission text it proposes to use.
7. Support a **dry run** that performs every possible check without opening Blackboard or submitting.
8. Ask before opening or controlling the Blackboard tab.
9. Require the student to authenticate personally; never request, read, store, type, or expose credentials.
10. Navigate only to this homework's submission page and stage the required files and repository URL.
11. Stop immediately before the final, irreversible submission action and show the student exactly what will be submitted.
12. Require explicit confirmation at that point. A prior general approval is not sufficient.
13. After confirmation, complete the submission, verify the confirmation page or receipt, and report the result.
14. Do not commit Blackboard screenshots, receipts, browser data, or personal information to the public repository.

Review the current [GitHub agent skills documentation](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/add-skills). Reload and inspect the skill with `/skills reload`, `/skills list`, and `/skills info blackboard-submission`.

Commit the skill and final notebook changes as:

```text
feat: add Blackboard submission skill
```

Push the branch. Then return to your repository's default branch, merge
`agent-work` **without squashing**, and push the default branch. Resolve and
review any conflict rather than asking Git to discard one side. Run the tests
again after the merge. The default branch, working tree, and remote must agree
before you create the submission archive.

### Part 9 laboratory record

Paste your complete `SKILL.md` here as a fenced code block.

**Answer**

```markdown
---
name: blackboard-submission
description: Preflight, package, and submit CSCI 6032 Homework 2 (repository gbradham/csci6032-hw2-gbradham) to Blackboard. Supports a dry run that performs every local check without opening a browser. Use only when the student asks to dry-run or submit this homework.
---

# Blackboard submission for CSCI 6032 Homework 2

Constants:

- Expected remote: `https://github.com/gbradham/csci6032-hw2-gbradham.git`
- Repository URL to submit: `https://github.com/gbradham/csci6032-hw2-gbradham`
- Archive: `csci6032-hw2-gbradham.tar.gz` (written to `dist/`, which is git-ignored, so the tree stays clean and the archive is never committed)
- Required files: `CSCI6032_hw2.ipynb`, `README.md`, `AGENTS.md`, `sample.txt`,
  `Dockerfile`, `src/text_stats.py`, `tests/test_text_stats.py`,
  `.github/skills/blackboard-submission/SKILL.md`
- Submission text:
  > I used my blackboard-submission skill, reviewed the staged artifacts, explicitly
  > approved the final submission action, and verified Blackboard's confirmation.

Mode: if the student says "dry run" (or does not say "submit"), run steps 1–5 only
and never open or control a browser.

Every shell and browser action goes through the normal permission prompt. Do not
ask for, or rely on, preapproved tools.

## 1. Confirm the repository

Run `git rev-parse --show-toplevel` and `git remote get-url origin`. Stop if the
remote is not the expected remote above.

## 2. Preflight

Run and show the output of:

```bash
git branch --show-current
git status --porcelain
git log --oneline -8
git fetch origin
git status -sb
```

Check that every required file exists in `HEAD` with `git ls-tree -r --name-only HEAD`.

## 3. Stop conditions

Stop and report the problem (do not try to "fix" it silently) if any of these hold:

- the working tree is not clean (`git status --porcelain` prints anything);
- a required file is missing from `HEAD`;
- the branch is not the default branch (`main`), or `HEAD` differs from `origin/main`;
- `git grep -nIiE '(api[_-]?key|secret|token|password|BEGIN [A-Z ]*PRIVATE KEY|ghp_|gho_|github_pat_)' HEAD`
  finds anything that is not clearly documentation; show each hit to the student
  and let them decide;
- the tree contains browser data, `.env` files, credential directories, or
  Blackboard screenshots/receipts.

## 4. Confirm the branch is pushed

`git rev-parse HEAD` must equal `git rev-parse origin/main`. Stop otherwise and ask
the student to push; do not push yourself.

## 5. Build and show the archive

Build from the committed `HEAD` only (this excludes `.git`, untracked files, and
anything ignored):

```bash
mkdir -p dist && git archive --format=tar.gz --prefix=csci6032-hw2-gbradham/ \
  -o dist/csci6032-hw2-gbradham.tar.gz HEAD
tar -tzf dist/csci6032-hw2-gbradham.tar.gz
```

Stop if the listing contains `__pycache__`, `*.pyc`, `.git/`, credentials, or
unrelated files. Then show the student exactly:

- notebook: `CSCI6032_hw2.ipynb` (at commit `<HEAD sha>`)
- archive: `dist/csci6032-hw2-gbradham.tar.gz` and its size
- repository URL
- submission text

**Dry run ends here.** Report "dry run complete" and the list of any problems.

## 6. Ask before touching the browser

Ask: "May I use the connected Blackboard tab now?" Wait for a clear yes.

## 7. Student authenticates

Ask the student to log in to Blackboard themselves. Never request, read, type,
store, or display credentials, MFA codes, or cookies. Continue only after the
student says they are logged in.

## 8. Navigate and stage

Navigate only to the CSCI 6032 Homework 2 submission page. Attach the notebook and
the archive, enter the repository URL and the submission text. Do not open other
courses, grades, messages, or unrelated pages.

## 9. Stop before the irreversible action

Do not click Submit. Show the student what is staged on the page: file names,
the repository URL, and the comment text. Ask: "Submit these exact items now? (yes/no)".
A previous general approval does not count. Only the literal answer to this
question counts.

## 10. Submit and verify

After an explicit "yes", click the final submit control once. Read the
confirmation page or receipt and report the confirmation details to the student.
If the automation cannot click safely, tell the student to click it themselves
and verify the confirmation together.

## Never

- commit or save Blackboard screenshots, receipts, page contents, or browser data to the repository;
- change Git history, commit, or push;
- submit during a dry run.
```

Check that every required file exists in `HEAD` with `git ls-tree -r --name-only HEAD`.

## 3. Stop conditions

Stop and report the problem (do not try to "fix" it silently) if any of these hold:

- the working tree is not clean (`git status --porcelain` prints anything);
- a required file is missing from `HEAD`;
- the branch is not the default branch (`main`), or `HEAD` differs from `origin/main`;
- `git grep -nIiE '(api[_-]?key|secret|token|password|BEGIN [A-Z ]*PRIVATE KEY|ghp_|gho_|github_pat_)' HEAD`
  finds anything that is not clearly documentation; show each hit to the student
  and let them decide;
- the tree contains browser data, `.env` files, credential directories, or
  Blackboard screenshots/receipts.

## 4. Confirm the branch is pushed

`git rev-parse HEAD` must equal `git rev-parse origin/main`. Stop otherwise and ask
the student to push; do not push yourself.

## 5. Build and show the archive

Build from the committed `HEAD` only (this excludes `.git`, untracked files, and
anything ignored):

```bash
git archive --format=tar.gz --prefix=csci6032-hw2-gbradham/ \
  -o ../csci6032-hw2-gbradham.tar.gz HEAD
tar -tzf ../csci6032-hw2-gbradham.tar.gz
```

Stop if the listing contains `__pycache__`, `*.pyc`, `.git/`, credentials, or
unrelated files. Then show the student exactly:

- notebook: `CSCI6032_hw2.ipynb` (at commit `<HEAD sha>`)
- archive: `../csci6032-hw2-gbradham.tar.gz` and its size
- repository URL
- submission text

**Dry run ends here.** Report "dry run complete" and the list of any problems.

## 6. Ask before touching the browser

Ask: "May I use the connected Blackboard tab now?" Wait for a clear yes.

## 7. Student authenticates

Ask the student to log in to Blackboard themselves. Never request, read, type,
store, or display credentials, MFA codes, or cookies. Continue only after the
student says they are logged in.

## 8. Navigate and stage

Navigate only to the CSCI 6032 Homework 2 submission page. Attach the notebook and
the archive, enter the repository URL and the submission text. Do not open other
courses, grades, messages, or unrelated pages.

## 9. Stop before the irreversible action

Do not click Submit. Show the student what is staged on the page: file names,
the repository URL, and the comment text. Ask: "Submit these exact items now? (yes/no)".
A previous general approval does not count. Only the literal answer to this
question counts.

## 10. Submit and verify

After an explicit "yes", click the final submit control once. Read the
confirmation page or receipt and report the confirmation details to the student.
If the automation cannot click safely, tell the student to click it themselves
and verify the confirmation together.

## Never

- commit or save Blackboard screenshots, receipts, page contents, or browser data to the repository;
- change Git history, commit, or push;
- submit during a dry run.
```

**Skill loaded:** from the repository root, host Copilot (`copilot -p "…list the agent skills that are loaded…"`) reported:

```text
- blackboard-submission — Preflight, package, and submit CSCI 6032 Homework 2 to Blackboard; supports dry runs.
```

**Change after the first dry run:** the first version of the skill wrote the archive to `../` (outside the repository). In the dry run, Copilot's path verification refused every command that wrote outside the current directory (`Permission denied and could not request permission from user`), so it stopped before the archive step and didn't open Blackboard. I didn't want to grant access to the repository's parent directory, which is my home directory. Instead I changed the skill to write to `dist/` and added `dist/` to `.gitignore`. `git archive HEAD` never includes untracked files, so the archive can't contain itself, and the tree stays clean.


Describe one material change you made after reviewing the agent's draft of the skill.

**Answer**

This skill draft was written by an AI agent (Claude Code). When I reviewed it, the main thing I checked was how the archive gets built. Packaging the working directory (`tar -czf … --exclude=.git .`) would include untracked files, `__pycache__`, and uncommitted edits, so the archive could differ from what's on GitHub. The skill uses `git archive … HEAD` instead, requires `HEAD == origin/main`, and stops when the listing contains `__pycache__`/`.pyc`. That last check is what exposes the bytecode committed in Part 4.

> **TODO (you):** describe one change *you* made to this draft after reading it (e.g., a stricter secret scan, a different archive location, wording of the final confirmation).

Why must the skill ask again immediately before the final submission action?

**Answer**

The earlier approvals (running the skill, opening the tab) were given before I could see what was actually staged on the page: wrong file, wrong assignment, a stale archive, or a typo in the URL. Submission is irreversible and outward-facing. It can't be undone with `git restore`, and it may count as my final attempt. A fresh, specific confirmation at the last moment makes me check the real staged state rather than the agent's description of it. It also guards against the agent drifting into the final click, or being prompted into it, while it only has general approval.

## Part 10. Dry-run and use the submission skill

Invoke `/blackboard-submission` in dry-run mode first. Resolve every reported problem. Inspect the archive contents yourself, confirm the repository is public and up to date, and confirm that no secret or private information is included.

Then invoke the skill for the real submission. Personally authenticate to Blackboard. Let the skill prepare the submission, inspect the staged files and text, and provide explicit confirmation only when they are correct. Verify the resulting Blackboard confirmation.

Do not modify the committed notebook afterward merely to record that submission succeeded; doing so would make the submitted archive differ from the repository checkpoint. Instead, enter this short statement in Blackboard alongside the repository URL:

```text
I used my blackboard-submission skill, reviewed the staged artifacts, explicitly
approved the final submission action, and verified Blackboard's confirmation.
```

If the automation cannot safely complete the final click, perform that click yourself, verify the result, and explain the limitation in the Blackboard comment. Safe human completion is preferable to bypassing a security boundary.

## Required Git history and repository contents

Your public repository must show, at minimum, these ordered checkpoints:

1. `checkpoint: repository before agent changes`
2. `docs: add agent safety instructions`
3. `feat: add agent-generated text statistics tool`
4. `checkpoint: add container before container agent changes`
5. `feat: add top-words option from container agent`
6. `feat: add Blackboard submission skill`

Equivalent additional commits are welcome. Do not squash these checkpoints.

Required contents:

```text
CSCI6032_hw2.ipynb
README.md
AGENTS.md
sample.txt
Dockerfile
src/
  text_stats.py
tests/
  test_text_stats.py
.github/
  skills/
    blackboard-submission/
      SKILL.md
```

Your project-level MCP configuration may be committed only if it contains no secret, personal path, browser data, or extension token. Otherwise, document its structure in the notebook and exclude it from the repository.

## Final reflection (5 points)

Answer each question in a short paragraph.

1. Where did Git provide protection, and where did it not?
2. How did the host sandbox and Docker container differ as boundaries?
3. Which agent output required the most human judgment?
4. What did the agent do that saved you time?
5. What would you change before allowing an agent to work on a valuable research repository?

**Answer**

1. **Git:** Git protected the **repository contents**. Every agent phase started from a pushed checkpoint, so any bad edit could be diffed and reverted, and GitHub held an off-machine copy. Git gave no protection for anything outside the repo (home directory, credentials, other projects, the browser session) or for side effects like network calls or form submissions. Git also can't un-publish a secret once it's been pushed. It only protects what's committed. Uncommitted work and ignored files are exposed.

2. **Host sandbox vs. Docker:** the Copilot local sandbox is a policy layer around my own user account. It depends on correct configuration and still runs on my real file system. The Docker container is a separate environment in which the host simply isn't visible except for one explicit mount. The sandbox limits what the agent is *allowed* to touch. The container limits what it can *see*. The container was the stronger boundary, and the sandbox was more convenient.

3. **Most human judgment:** the submission skill and the review of generated code. Code that passes tests can still commit build artifacts, like the `.pyc` files here. And the skill's archive step looked right while packaging the wrong tree. Both needed me to check the actual state (`git ls-tree`, `tar -tzf`) rather than the agent's summary.

4. **Time saved:** a correct, tested, standard-library-only CLI with Unicode-aware tests and README docs in one pass. Drafting the Dockerfile's apt keyring setup for `gh` and the layer cleanup would also have taken me a while from the docs.

5. **Before a valuable research repo:** run agents only in a container or VM with no credentials and restricted network egress. Use a separate branch or worktree and never let the agent push. Add a `.gitignore` and pre-commit secret scanning. Make sure there are off-site backups beyond Git (data files are often not in Git). Require tests or reproducibility checks before any merge. Keep write access to raw data read-only.

> **TODO (you):** rewrite these in your own words where they don't match your experience.

## Grading summary

| Component | Points |
|---|---:|
| Access and preparation | 5 |
| Host Copilot installation and Hello World | 10 |
| Repository and initial checkpoint | 10 |
| Agent instructions and host boundary | 10 |
| Host-agent code generation and review | 15 |
| Docker installation | 10 |
| Linux agent development image | 15 |
| Agent work inside the container | 10 |
| Browser tools | 5 |
| Submission skill | 5 |
| Final reflection | 5 |
| **Total** | **100** |

The real Blackboard submission is required for the homework to be received. If browser automation fails after a documented good-faith attempt, submit manually and document the limitation in the Blackboard comment.

## Documentation references

These links were checked when the assignment was prepared. Tools change quickly; use the current official page when its interface differs from this notebook.

- [Installing GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/install-copilot-cli)
- [Authenticating GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/authenticate-copilot-cli)
- [Using local sandboxing](https://docs.github.com/en/copilot/how-tos/cloud-and-local-sandboxes/using-local-sandboxing)
- [Allowing and denying Copilot CLI tools](https://docs.github.com/en/copilot/how-tos/copilot-cli/use-copilot-cli/allowing-tools)
- [Adding agent skills](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/add-skills)
- [Adding MCP servers](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/add-mcp-servers)
- [Docker Desktop](https://docs.docker.com/desktop/)
- [Docker Engine installation](https://docs.docker.com/engine/install/)
- [Docker bind mounts](https://docs.docker.com/engine/storage/bind-mounts/)
- [Playwright MCP](https://github.com/microsoft/playwright-mcp)
- [Playwright MCP Bridge extension](https://github.com/microsoft/playwright/tree/main/packages/extension)